# 06 — Singular Value Decomposition (SVD)
## The Swiss Army Knife of Linear Algebra

> **Prerequisites:** Notebooks 01–05  
> **Difficulty:** ⭐⭐⭐⭐☆ Intermediate-Advanced  
> **Running example:** Image compression, PCA, recommender systems

---

## What problem does SVD solve?

Eigendecomposition (Notebook 05) was great, but it had a limitation: **it only works on square matrices**.

Your dataset X is usually **not square** — it's (1000 samples × 20 features).

**SVD works on any matrix of any shape.**

---

## The big idea: Every matrix is really 3 simple operations

> **A = U Σ Vᵀ**

| Part | Shape | What it does |
|---|---|---|
| **U** | m×m | Rotates/reflects the output space |
| **Σ** (Sigma) | m×n | Scales — stretches along each direction |
| **Vᵀ** | n×n | Rotates/reflects the input space |

**Analogy:** Think of folding a piece of paper and stretching it:
1. Vᵀ: orient the paper (rotate input)
2. Σ: pull/push it along each axis (scale)
3. U: orient the result (rotate output)

The stretching amounts (the diagonal of Σ) are the **singular values** σ₁ ≥ σ₂ ≥ ... ≥ 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPUTING SVD
# ─────────────────────────────────────────────────────────

A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.],
              [10.,11.,12.]])   # shape (4, 3) — NOT square!

print(f"Matrix A, shape {A.shape}:")
print(A)
print()

# np.linalg.svd returns three things: U, s (not Sigma), Vt
# full_matrices=False → 'economy' SVD, smaller matrices, more practical
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f"U  shape: {U.shape}   ← left singular vectors (columns = directions in output space)")
print(f"s  shape: {s.shape}   ← singular values (1D array, not a diagonal matrix)")
print(f"Vt shape: {Vt.shape}  ← right singular vectors transposed (rows = directions in input space)")
print()
print(f"Singular values: {s.round(4)}")
print()
print("Note: np.linalg.svd gives 's' as a 1D array (not a matrix).")
print("To get the Σ matrix: np.diag(s)")
Sigma = np.diag(s)
print(f"Σ =\n{Sigma.round(4)}")
print()

# Reconstruct A to verify: A = U Σ Vt
A_reconstructed = U @ Sigma @ Vt
print(f"Reconstruction A = U @ Σ @ Vt, error: {np.linalg.norm(A - A_reconstructed):.2e}")
print("(near zero = perfect ✓)")

---
## 2. What Do the Singular Values Mean?

### Plain English
The **singular values** σ₁ ≥ σ₂ ≥ ... ≥ 0 measure the **importance** of each direction.

- Large σ → this direction carries a lot of information (a lot of "energy")
- Small σ → this direction barely matters
- σ = 0 → this direction carries ZERO information (the matrix is rank-deficient)

### Rank from singular values
**Rank of A** = number of non-zero singular values.

### Relationship to eigenvalues
The singular values of A are the **square roots of the eigenvalues of AᵀA**:
> σᵢ = √λᵢ(AᵀA)

In [ ]:
# ─────────────────────────────────────────────────────────
# SINGULAR VALUES AND RANK
# ─────────────────────────────────────────────────────────

print("=== Singular values reveal rank ===")
print()

# Full rank matrix: 2 non-zero singular values
A_full = np.array([[1., 2.], [3., 4.], [5., 6.]])  # shape (3,2)
_, s_full, _ = np.linalg.svd(A_full)
print(f"Full rank matrix (3×2):")
print(f"  Singular values: {s_full.round(4)}")
print(f"  Rank: {np.sum(s_full > 1e-10)}  (both non-zero → full rank = 2)")

print()
# Rank-1 matrix: row 2 = 2 × row 1, so only 1 independent direction
A_rank1 = np.array([[1., 2., 3.],
                    [2., 4., 6.]])   # row 2 = 2 × row 1
_, s_rank1, _ = np.linalg.svd(A_rank1)
print(f"Rank-1 matrix (row2 = 2×row1):")
print(f"  Singular values: {s_rank1.round(6)}")
print(f"  Rank: {np.sum(s_rank1 > 1e-10)}  (only 1 non-zero → rank = 1)")
print(f"  Interpretation: all rows are multiples of each other")
print(f"  → the matrix contains only 1 'direction' of information")

print()
print("=== Relationship to eigenvalues ===")
A = np.random.randn(5, 3)
_, s_svd, _ = np.linalg.svd(A, full_matrices=False)
eig_AtA = np.linalg.eigvalsh(A.T @ A)[::-1]  # eigenvalues of AᵀA, sorted descending
print(f"σ from SVD:               {s_svd.round(4)}")
print(f"√(eigenvalues of AᵀA):    {np.sqrt(np.abs(eig_AtA)).round(4)}")
print(f"They match: {np.allclose(s_svd, np.sqrt(np.abs(eig_AtA)))}")

---
## 3. Low-Rank Approximation — The Key Power of SVD

### Plain English
SVD lets you **compress** a matrix by keeping only the most important directions.

The **rank-k approximation** keeps only the top k singular values:
> **Aₖ = U[:, :k] × Σₖ × Vᵀ[:k, :]**

This is the **best possible rank-k approximation** (Eckart-Young theorem — it's proven to be optimal).

### How much do we keep?
**Energy fraction** = (σ₁² + σ₂² + ... + σₖ²) / (σ₁² + ... + σₙ²)

### Real-world applications
| Application | What gets compressed |
|---|---|
| Image compression | Keep top k singular vectors instead of all pixels |
| Netflix recommendations | User-movie rating matrix, find latent factors |
| Topic modeling (LSA) | Document-word matrix, find hidden topics |
| PCA | Keep top k principal directions of data |

In [ ]:
# ─────────────────────────────────────────────────────────
# LOW-RANK APPROXIMATION
# ─────────────────────────────────────────────────────────

print("=== Low-rank approximation: keep only top k singular values ===")
print()

# Create a matrix with known low-rank structure + noise
# (Like real data: true pattern + measurement noise)
true_rank = 3
U_true = np.random.randn(30, true_rank)
V_true = np.random.randn(20, true_rank)
A_true = U_true @ V_true.T             # rank 3 structure
noise  = 0.3 * np.random.randn(30, 20) # noise on top
A = A_true + noise

U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f"True rank: {true_rank}")
print(f"Top 8 singular values: {s[:8].round(3)}")
print("Notice the big drop after the 3rd value!")
print()

# Compare approximations with different k values
ks = [1, 2, 3, 5, 10, 20]
for k in ks:
    # Rank-k approximation: multiply only first k columns/rows
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]   # same as U[:,:k] @ diag(s[:k]) @ Vt[:k,:]
    error = np.linalg.norm(A - A_k, 'fro') / np.linalg.norm(A, 'fro')
    energy = (s[:k]**2).sum() / (s**2).sum()
    # Storage: original needs 30×20=600 numbers; rank-k needs k*(30+20+1)
    compression = (30 * 20) / (k * (30 + 20 + 1))
    print(f"k={k:2d}: error={error:.3f}, energy={energy:.1%}, compression={compression:.1f}x")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUAL: image compression with SVD
# ─────────────────────────────────────────────────────────

# Create a synthetic image-like matrix with structure
image = np.zeros((60, 60))
for _ in range(8):  # 8 "blobs" of varying size and brightness
    cx, cy = np.random.randint(10, 50, 2)
    r = np.random.randint(5, 15)
    y, x = np.ogrid[:60, :60]
    image += np.random.uniform(0.3, 1.0) * np.exp(-((x-cx)**2 + (y-cy)**2) / (2*r**2))
image = (image / image.max() * 255).astype(float)

U, s, Vt = np.linalg.svd(image, full_matrices=False)
total_energy = (s**2).sum()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
ks = [1, 3, 5, 10, 20, 30, 45, 60]

for ax, k in zip(axes.flat, ks):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    energy = (s[:k]**2).sum() / total_energy
    original_size = 60 * 60
    compressed_size = k * (60 + 60 + 1)
    compression = original_size / compressed_size
    
    ax.imshow(np.clip(A_k, 0, 255), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'k={k}  ({energy:.0%} energy)\n{compression:.1f}x compression', fontsize=9)
    ax.axis('off')

plt.suptitle('SVD Image Compression\nSmaller k = fewer singular values = more compressed but lower quality',
             fontsize=11)
plt.tight_layout()
plt.show()
print("Which k gives acceptable quality with good compression?")

---
## 4. SVD and PCA — Two Names for the Same Idea

### How PCA relates to SVD
PCA finds the **directions of maximum variance** in data.
SVD of the centered data gives exactly those directions.

**The recipe:**
1. Center the data: `X_c = X - X.mean(axis=0)`
2. Compute SVD: `U, s, Vt = np.linalg.svd(X_c, full_matrices=False)`
3. **Columns of V (rows of Vᵀ) = principal component directions**
4. Variance explained by PC_i = σᵢ² / (n−1)

In [ ]:
# ─────────────────────────────────────────────────────────
# SVD = PCA (when applied to centered data)
# ─────────────────────────────────────────────────────────

print("=== PCA via SVD ===")
print()

# 3-feature dataset with strong correlation
t = np.random.randn(200)
X = np.column_stack([
    t + 0.2*np.random.randn(200),
    2*t + 0.2*np.random.randn(200),
    0.5*t + 0.2*np.random.randn(200),
])
X -= X.mean(axis=0)  # center

print(f"Data shape: {X.shape}  (200 samples × 3 features)")
print()

U, s, Vt = np.linalg.svd(X, full_matrices=False)
n = X.shape[0]

# Variance explained
variance_explained = s**2 / (n - 1)
total_var = variance_explained.sum()
pct = variance_explained / total_var * 100

print(f"Singular values: {s.round(3)}")
print(f"Variance explained: {variance_explained.round(3)}")
print(f"Percentage: PC1={pct[0]:.1f}%, PC2={pct[1]:.1f}%, PC3={pct[2]:.1f}%")
print()
print("PC1 captures almost all the variance — data is essentially 1D!")
print("(Makes sense: all features are based on the same variable t)")
print()

# Project onto top 2 PCs
X_2d = X @ Vt.T[:, :2]   # project to 2D
print(f"Original:  {X.shape}   →   PCA 2D:  {X_2d.shape}")
print("Reduced from 3D to 2D while keeping {:.0f}% of variance".format((pct[:2].sum())))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(X[:,0], X[:,1], alpha=0.3, s=15, c=t, cmap='viridis')
ax1.set_xlabel('Feature 1'); ax1.set_ylabel('Feature 2')
ax1.set_title('Original Data (first 2 features shown)', fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.scatter(X_2d[:,0], X_2d[:,1], alpha=0.3, s=15, c=t, cmap='viridis')
ax2.set_xlabel(f'PC1 ({pct[0]:.1f}% var)'); ax2.set_ylabel(f'PC2 ({pct[1]:.1f}% var)')
ax2.set_title('After PCA (via SVD)', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.colorbar(ax2.collections[0], ax=ax2, label='Underlying factor t')
plt.suptitle('SVD reveals the true underlying structure in data', fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

```python
# The three outputs of SVD
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Reconstruct A exactly
A_back = U @ np.diag(s) @ Vt

# Rank-k approximation (compression / denoising)
k = 5
A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]

# PCA: project data onto top k directions
X_pca = X_centered @ Vt.T[:, :k]   # OR equivalently: X_centered @ V[:, :k]
```

| Concept | Meaning |
|---|---|
| U (columns) | Output directions ("what it looks like after transform") |
| s (singular values) | How much each direction matters (strength/energy) |
| Vt (rows) | Input directions ("what to look for in input") |
| Low-rank approx | Keep top k components → compression & denoising |
| PCA connection | SVD of centered data = PCA |

**Next: Notebook 07 — Vector Spaces, Span, Basis, Rank & Null Space**